# ML Zoomcamp 2026 – Homework 6

## Preliminaries

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline

from pathlib import Path
from pprint import pprint
from IPython.display import display

In [2]:
data_path = Path("./data/car_fuel_efficiency_2026.csv").resolve()

df = pd.read_csv(data_path)
print(df.shape)
df.head()

(10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 1.1 MB


In [4]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [5]:
df.nunique()

model_year              50
origin                   3
fuel_type                3
drivetrain               3
num_doors                4
engine_displacement    127
num_cylinders            5
horsepower             153
vehicle_weight         176
acceleration            93
fuel_efficiency_mpg    188
dtype: int64

In [6]:
df = df.fillna(0)
df.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [7]:
categorical_features = list(df.dtypes[df.dtypes == "str"].index)
print(categorical_features)

for c in categorical_features:
    display(df[c].unique())

['origin', 'fuel_type', 'drivetrain']


<ArrowStringArray>
['Europe', 'Asia', 'USA']
Length: 3, dtype: str

<ArrowStringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str

<ArrowStringArray>
['Front-wheel drive', 'Rear-wheel drive', 'All-wheel drive']
Length: 3, dtype: str

In [8]:
for c in categorical_features:
    df[c] = df[c].str.lower().str.replace(' ', '_')

for c in categorical_features:
    display(df[c].unique())

<ArrowStringArray>
['europe', 'asia', 'usa']
Length: 3, dtype: str

<ArrowStringArray>
['gasoline', 'diesel', 'hybrid']
Length: 3, dtype: str

<ArrowStringArray>
['front-wheel_drive', 'rear-wheel_drive', 'all-wheel_drive']
Length: 3, dtype: str

In [9]:
SEED = 1

# Split data
df_full_train, df_test = train_test_split(
    df, 
    test_size=0.2, 
    random_state=SEED
)
df_train, df_val = train_test_split(
    df_full_train, 
    test_size=0.25, 
    random_state=SEED
)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

# Prepare target
y_train = df_train["fuel_efficiency_mpg"].values
y_val = df_val["fuel_efficiency_mpg"].values
y_test = df_test["fuel_efficiency_mpg"].values

# Remove target from set
df_train.drop("fuel_efficiency_mpg", axis=1, inplace=True)
df_val.drop("fuel_efficiency_mpg", axis=1, inplace=True)
df_test.drop("fuel_efficiency_mpg", axis=1, inplace=True)

df_train.shape, df_val.shape, df_test.shape, df_full_train.shape

((6000, 10), (2000, 10), (2000, 10), (8000, 11))

In [10]:
train_dict = df_train.to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

In [11]:
dv.get_feature_names_out()

array(['acceleration', 'drivetrain=all-wheel_drive',
       'drivetrain=front-wheel_drive', 'drivetrain=rear-wheel_drive',
       'engine_displacement', 'fuel_type=diesel', 'fuel_type=gasoline',
       'fuel_type=hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=asia', 'origin=europe', 'origin=usa',
       'vehicle_weight'], dtype=object)

## Q1

In [12]:
dtr = DecisionTreeRegressor(max_depth=1, random_state=SEED)
dtr.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``m

In [13]:
print(
    export_text(dtr, feature_names=list(dv.get_feature_names_out()))
)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Q2

In [14]:
rtr = RandomForestRegressor(n_estimators=10, random_state=SEED)
rtr.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of 

In [15]:
y_pred = rtr.predict(X_train)
rmse_train = root_mean_squared_error(y_train, y_pred)
rmse_train

0.7714290526894441

In [16]:
val_dict = df_val.to_dict(orient="records")
X_val = dv.transform(val_dict)

y_pred = rtr.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred)
rmse_val

1.8336350509302553

In [17]:
model_rtr = make_pipeline(
    DictVectorizer(sparse=False),
    RandomForestRegressor(n_estimators=10, random_state=SEED)
)

model_rtr.fit(train_dict, y_train)

y_pred = model_rtr.predict(train_dict)
print(f"train -> {root_mean_squared_error(y_train, y_pred)}")

y_pred = model_rtr.predict(val_dict)
print(f"val -> {root_mean_squared_error(y_val, y_pred)}")

train -> 0.7714290526894441
val -> 1.8336350509302553


## Q3

In [18]:
estimators_counts = [10, 50, 100, 150]
rmse_scores = {}

for e in tqdm(estimators_counts):
    model = make_pipeline(
        DictVectorizer(sparse=False),
        RandomForestRegressor(n_estimators=e, random_state=SEED)
    )

    model.fit(train_dict, y_train)

    y_pred = model.predict(val_dict)
    rmse_scores[e] = root_mean_squared_error(y_val, y_pred)

for e, rmse in rmse_scores.items():
    print(f"n_estimators={e} --> {rmse:.5f}")

  0%|          | 0/4 [00:00<?, ?it/s]

n_estimators=10 --> 1.83364
n_estimators=50 --> 1.76963
n_estimators=100 --> 1.76752
n_estimators=150 --> 1.76753


## Q4

In [21]:
estimators_counts = [10, 50, 100, 150]
max_depths = [10, 15, 20, 25, 30, 40, 50] # Added additional values for double checking
rmse_scores = []

for d in tqdm(max_depths):
    for e in estimators_counts:
        model = make_pipeline(
            DictVectorizer(sparse=False),
            RandomForestRegressor(
                n_estimators=e,
                max_depth=d,
                random_state=SEED
            )
        )

        model.fit(train_dict, y_train)

        y_pred = model.predict(val_dict)

        rmse_scores.append({
            "max_depth": d,
            "n_estimators": e,
            "rmse": root_mean_squared_error(y_val, y_pred)
        })

  0%|          | 0/7 [00:00<?, ?it/s]

In [22]:
df_rmse_scores = pd.DataFrame(rmse_scores)

df_rmse_scores.groupby("max_depth")["rmse"].agg("mean")

max_depth
10    1.756866
15    1.784782
20    1.782678
25    1.787373
30    1.784919
40    1.784576
50    1.784576
Name: rmse, dtype: float64

## Q5

In [23]:
model = make_pipeline(
    DictVectorizer(sparse=False),
    RandomForestRegressor(
        n_estimators=10,
        max_depth=20,
        random_state=SEED
    )
)

model.fit(train_dict, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('randomforestregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value
"feature_names_ feature_names_: listA list of length n_features containing the feature names (e.g., ""f=ham""and ""f=spam"").",list,"['ac...on', 'dr...ve', 'dr...ve', 'dr...ve', ...]"
vocabulary_ vocabulary_: dictA dictionary mapping feature names to feature indices.,dict,"{'ac...on': 0, 'dr...ve': 1, 'dr...ve': 2, 'dr...ve': 3, ...}"


In [37]:
model[-1].feature_importances_

array([0.05546622, 0.00787275, 0.0200048 , 0.00563607, 0.06100789,
       0.02112758, 0.16071286, 0.01168541, 0.07825357, 0.29582159,
       0.00693892, 0.01972375, 0.00647862, 0.00338878, 0.04223247,
       0.20364873])

In [38]:
model[0].get_feature_names_out()

array(['acceleration', 'drivetrain=all-wheel_drive',
       'drivetrain=front-wheel_drive', 'drivetrain=rear-wheel_drive',
       'engine_displacement', 'fuel_type=diesel', 'fuel_type=gasoline',
       'fuel_type=hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=asia', 'origin=europe', 'origin=usa',
       'vehicle_weight'], dtype=object)

In [45]:
importances = pd.DataFrame(
    zip(model[0].get_feature_names_out(), model[-1].feature_importances_),
    columns=["features", "importance"]
)

importances

,features,importance
0,acceleration,0.055466
1,drivetrain=all-wheel_drive,0.007873
2,drivetrain=front-wheel_drive,0.020005
3,drivetrain=rear-wheel_drive,0.005636
4,engine_displacement,0.061008
5,fuel_type=diesel,0.021128
6,fuel_type=gasoline,0.160713
7,fuel_type=hybrid,0.011685
8,horsepower,0.078254
9,model_year,0.295822


In [47]:
importances.sort_values(by="importance", ascending=False)

,features,importance
9,model_year,0.295822
15,vehicle_weight,0.203649
6,fuel_type=gasoline,0.160713
8,horsepower,0.078254
4,engine_displacement,0.061008
0,acceleration,0.055466
14,origin=usa,0.042232
5,fuel_type=diesel,0.021128
2,drivetrain=front-wheel_drive,0.020005
11,num_doors,0.019724


## Q6

In [49]:
train_dict = df_train.to_dict(orient="records")
val_dict = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [50]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [52]:
xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": SEED,
    "verbosity": 1
}

model_xgb = xgb.train(
    xgb_params, 
    dtrain, 
    num_boost_round=100
)

y_pred = model_xgb.predict(dval)
root_mean_squared_error(y_val, y_pred)

1.8264579799904583

In [53]:
xgb_params = {
    "eta": 0.1,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": SEED,
    "verbosity": 1
}

model_xgb = xgb.train(
    xgb_params, 
    dtrain, 
    num_boost_round=100
)

y_pred = model_xgb.predict(dval)
root_mean_squared_error(y_val, y_pred)

1.724810059827976